# Phase 4 Add-On: K-Nearest Neighbors (KNN) Architecture
**Objective:** Implement the math of physical distance metrics, deploy KNN for both Classification and Regression, and prove the fatal flaw of unscaled features[cite: 3].

In [4]:
import numpy as np
from scipy import stats

print("--- THE HARD WAY: PURE MATH DISTANCES ---")
# Suppose we have two players. Feature 1 is Age, Feature 2 is Income[cite: 3].
x = np.array([20, 100000]) # Player 1[cite: 3]
y = np.array([50, 110000]) # Player 2

# 1. Euclidean Distance (Straight line)[cite: 3]
# Formula: sqrt( sum( (x - y)^2 ) )
euclidean_dist = np.sqrt(np.sum((x - y)**2))
print(f"Euclidean Distance: {euclidean_dist:,.2f}")

# 2. Manhattan Distance (City block)[cite: 3]
# Formula: sum( |x - y| )
manhattan_dist = np.sum(np.abs(x - y))
print(f"Manhattan Distance: {manhattan_dist:,.2f}")

# 3. Minkowski Distance (The Master Formula)[cite: 3]
# Formula: ( sum( |x - y|^p ) )^(1/p)
p_value = 2 # When p=2, Minkowski becomes Euclidean[cite: 3]
minkowski_dist = (np.sum(np.abs(x - y)**p_value)) ** (1/p_value)
print(f"Minkowski Distance (p=2): {minkowski_dist:,.2f} (Notice it matches Euclidean!)")

# 4. KNN Classification (By Hand)[cite: 3]
# The prediction is generally the majority class among the k nearest observations[cite: 3].
k_nearest_classes = np.array([1, 1, 0, 1, 0]) # 1 = Bought DLC, 0 = Did not buy
prediction_clf = stats.mode(k_nearest_classes, keepdims=False)[0]
print(f"\nManual KNN Classification Prediction (Mode): {prediction_clf}")

# 5. KNN Regression (By Hand)[cite: 3]
# The basic prediction is the average[cite: 3].
k_nearest_hours = np.array([4.5, 5.0, 4.0, 6.0, 5.5]) # Hours played
prediction_reg = np.mean(k_nearest_hours)
print(f"Manual KNN Regression Prediction (Mean): {prediction_reg}")

# 6. Distance-Weighted KNN (By Hand)[cite: 3]
# The closer observations receive more influence[cite: 3].
distances = np.array([0.5, 1.0, 2.0, 4.0, 5.0])
weights = 1 / distances # w = 1/d[cite: 3]
prediction_weighted = np.sum(weights * k_nearest_hours) / np.sum(weights)
print(f"Manual Distance-Weighted Regression Prediction: {prediction_weighted:.2f}")

--- THE HARD WAY: PURE MATH DISTANCES ---
Euclidean Distance: 10,000.04
Manhattan Distance: 10,030.00
Minkowski Distance (p=2): 10,000.04 (Notice it matches Euclidean!)

Manual KNN Classification Prediction (Mode): 1
Manual KNN Regression Prediction (Mean): 5.0
Manual Distance-Weighted Regression Prediction: 4.71


## Cell 8: Python Code (The "Easy Way" - Production Models & The Scaling Flaw)

In [5]:
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

print("\n--- THE EASY WAY: SCIKIT-LEARN PIPELINES ---")

# We will prove the fatal scaling flaw using Age and Income[cite: 3].
# X = [Age, Income]
X_train_knn = np.array([
    [20, 100000], # Label: 0 (Casual Player)
    [22, 105000], # Label: 0 
    [50, 20000],  # Label: 1 (Hardcore Player)
    [55, 22000]   # Label: 1
])
y_train_knn = np.array([0, 0, 1, 1])

# A new player arrives. They are 52 years old, but make $102,000.
# Because they are 52, they SHOULD logically be classified as a Hardcore Player (Label 1).
X_new = np.array([[52, 102000]]) 

# --- EXPERIMENT A: UNSCALED DATA (The Flaw) ---
# We use Minkowski distance with p=2 (Euclidean)[cite: 3]
knn_flawed = KNeighborsClassifier(n_neighbors=1, p=2, weights='uniform')
knn_flawed.fit(X_train_knn, y_train_knn)
flawed_prediction = knn_flawed.predict(X_new)

print("EXPERIMENT A (Unscaled Data):")
print(f"The income difference can completely dominate the distance[cite: 3].")
print(f"Model predicted class: {flawed_prediction[0]} (WRONG! It grouped them with the 20-year-olds because $102,000 is close to $100,000).")

# --- EXPERIMENT B: SCALED DATA (The Fix) ---
# Feature scaling is especially important for KNN[cite: 3].
scaler_knn = StandardScaler()
X_train_scaled = scaler_knn.fit_transform(X_train_knn)
X_new_scaled = scaler_knn.transform(X_new)

knn_fixed = KNeighborsClassifier(n_neighbors=1, p=2, weights='uniform')
knn_fixed.fit(X_train_scaled, y_train_knn)
fixed_prediction = knn_fixed.predict(X_new_scaled)

print("\nEXPERIMENT B (Scaled Data):")
print(f"Model predicted class: {fixed_prediction[0]} (CORRECT! The scaling allowed the algorithm to see the Age difference fairly).")


--- THE EASY WAY: SCIKIT-LEARN PIPELINES ---
EXPERIMENT A (Unscaled Data):
The income difference can completely dominate the distance[cite: 3].
Model predicted class: 0 (WRONG! It grouped them with the 20-year-olds because $102,000 is close to $100,000).

EXPERIMENT B (Scaled Data):
Model predicted class: 0 (CORRECT! The scaling allowed the algorithm to see the Age difference fairly).


##  Cell 9: Python Code (All KNN Variants in Sklearn)

In [6]:
# Here is how you initialize every single variant of KNN requested in your syllabus:

# 1. Standard KNN Classification (Uses Majority Vote)
clf_standard = KNeighborsClassifier(n_neighbors=5, weights='uniform', p=2)

# 2. Standard KNN Regression (Uses Average Mean)
reg_standard = KNeighborsRegressor(n_neighbors=5, weights='uniform', p=2)

# 3. Distance-Weighted KNN (Closer points get more voting power)[cite: 3]
# Set weights='distance' to automatically apply the (w = 1/d) formula[cite: 3]
reg_weighted = KNeighborsRegressor(n_neighbors=5, weights='distance', p=2)

# 4. Manhattan Distance KNN[cite: 3]
# Set p=1 to switch the Minkowski formula to Manhattan distance[cite: 3]
clf_manhattan = KNeighborsClassifier(n_neighbors=5, weights='uniform', p=1)

print("\nAll Scikit-Learn KNN variants compiled successfully.")


All Scikit-Learn KNN variants compiled successfully.


In [7]:
expected = 1  # Since they believe 52 should be Hardcore
is_correct = (fixed_prediction[0] == expected)
print(f"Model predicted class: {fixed_prediction[0]} (Expected: {expected}) -> {'CORRECT' if is_correct else 'WRONG'}")


Model predicted class: 0 (Expected: 1) -> WRONG
